In [4]:
%pip install openpyxl

In [6]:
import piplite
await piplite.install("openpyxl")

In [8]:
import glob
import pandas as pd
import numpy as np

# 1. ЧТЕНИЕ CSV с заказами и Excel с ТОВАРАМИ
files = sorted(glob.glob("data/orders_2024_*.csv"))
print(files)
orders = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
products = pd.read_excel("data/products.xlsx", sheet_name="products")

# 2. ОБЪЕДИНЕНИЕ ЗАКАЗОВ И ТОВАРОВ, ПРИВЕДЕНИЕ ТИПОВ
df = orders.merge(products, on="product_id", how="left")
df["price"] = df["price"].astype("float32")
for col in ["order_id", "user_id", "product_id"]:
    df[col] = df[col].astype("int32")
df["category_id"] = df["category_id"].astype("Int32")  # допускает пропуски

# 3. ВЫРУЧКА ПО КОМБИНАЦИИ (user_id, category_id)
df["revenue"] = df["price"] * df["quantity"]
rev = df.groupby(["user_id", "category_id"])["revenue"].sum().reset_index()

# 4. ТОП-5 КАТЕГОРИЙ ДЛЯ КАЖДОГО ПОЛЬЗОВАТЕЛЯ ПО ВЫРУЧКЕ
top5 = (rev.sort_values(["user_id", "revenue"], ascending=[True, False])
           .groupby("user_id").head(5))
print(top5.head(10))

# 5. СОХРАНЕНИЕ В parquet (если не получится - в CSV)
try:
    top5.to_parquet("top5_categories_per_user.parquet", index=False)
except Exception as e:
    print("Parquet НЕДОСТУПЕН:", e)
    top5.to_csv("top5_categories_per_user.csv", index=False)

# ПРОВЕРКА: КОЛИЧЕСТВО СТРОК И ТИПЫ ДАННЫХ
print(df.shape)
print(df.dtypes)

['data/orders_2024_01.csv', 'data/orders_2024_02.csv', 'data/orders_2024_03.csv']
   user_id  category_id       revenue
4     1001            5  1.238283e+06
2     1001            3  1.212973e+06
0     1001            1  1.079367e+06
1     1001            2  5.109530e+05
3     1001            4  3.941912e+05
9     1002            5  9.024817e+05
7     1002            3  7.036556e+05
5     1002            1  4.306263e+05
6     1002            2  4.098305e+05
8     1002            4  3.834103e+05
Parquet НЕДОСТУПЕН: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.
(15000, 12)
order_id          